In [14]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [18]:
# 1. Charger les données
df = pd.read_csv("C:\\Users\\DELL\\Downloads\\data_set.csv", encoding="latin1", sep=";")

# 2. Définir les colonnes d'entrée (features)
print(df.head())

# Afficher la liste complète des colonnes
print(df.columns)


  ville_depart ville_arrivee  distance  hour   weekday  is_rain  is_rush_hour  \
0       Agadir         Rabat       550     11        1        0             0   
1   Casablanca     Marrakech       240      9        0        1             1   
2       Tanger            F?       320     18        5        0             1   
3        Oujda         Nador       130     16        6        1             1   
4    Marrakech        Agadir       250      8        2        0             1   

   trip_duration  
0            660  
1            300  
2            370  
3            150  
4            210  
Index(['ville_depart', 'ville_arrivee', 'distance', 'hour ', 'weekday',
       'is_rain', 'is_rush_hour', 'trip_duration'],
      dtype='object')


In [19]:
# retirer les espaces
df.columns = df.columns.str.strip()

# Définir les colonnes d'entrée (features)
features = ['distance', 'hour', 'weekday', 'is_rain', 'is_rush_hour']
X = df[features]

# Définir la cible
y = df['trip_duration']


In [20]:
# Supprimer les espaces autour des noms de colonnes
df.columns = df.columns.str.strip()

# Supprimer les lignes avec des valeurs manquantes
df = df.dropna()

# Définir les colonnes d'entrée (features)
features = ['distance', 'hour', 'weekday', 'is_rain', 'is_rush_hour']
X = df[features]

# Définir la cible (valeur à prédire)
y = df['trip_duration']

In [21]:
# Séparer en données d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
# Créer et entraîner le modèle
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [23]:
# Prédire sur les données de test
y_pred = model.predict(X_test)

# Afficher les prédictions et les vraies valeurs
print("Prédictions :", y_pred)
print("Valeurs réelles :", y_test.values)



Prédictions : [169.5085956  443.13636163 615.64032816]
Valeurs réelles : [165 330 660]


In [24]:
# Distances entre quelques villes (tu peux ajouter d'autres)
distances = {
    ('Agadir', 'Rabat'): 550,
    ('Casablanca', 'Marrakech'): 240,
    ('Tanger', 'Fès'): 320,
    ('Oujda', 'Nador'): 130,
    ('Marrakech', 'Essaouira'): 190
}

def get_distance(ville_depart, ville_arrivee):
    if (ville_depart, ville_arrivee) in distances:
        return distances[(ville_depart, ville_arrivee)]
    elif (ville_arrivee, ville_depart) in distances:
        return distances[(ville_arrivee, ville_depart)]
    else:
        print("Distance inconnue entre ces villes.")
        return None

In [25]:
# Définir un dictionnaire pour associer chaque trajet à un type de route (0=ville, 1=route, 2=autoroute)
# Exemple simple, à adapter selon tes trajets réels
type_route_par_trajet = {
    ('Agadir', 'Rabat'): 2,           # autoroute
    ('Casablanca', 'Marrakech'): 2,   # autoroute
    ('Tanger', 'Fès'): 1,             # route nationale
    ('Oujda', 'Nador'): 1,
    ('Marrakech', 'Essaouira'): 1,
}

# Vitesse moyenne par type de route (en km/h)
vitesses_par_type = {
    0: 40,   # ville
    1: 70,   # route nationale
    2: 100   # autoroute
}

# Fonction pour déterminer type_route pour chaque ligne
def get_type_route(row):
    trajet = (row['ville_depart'], row['ville_arrivee'])
    trajet_inverse = (row['ville_arrivee'], row['ville_depart'])
    if trajet in type_route_par_trajet:
        return type_route_par_trajet[trajet]
    elif trajet_inverse in type_route_par_trajet:
        return type_route_par_trajet[trajet_inverse]
    else:
        # Par défaut on met route nationale (1)
        return 1

In [26]:
#Appliquer la fonction pour créer la colonne 'type_route'
df['type_route'] = df.apply(get_type_route, axis=1)


In [27]:
# Créer la colonne vitesse en fonction du type_route
df['vitesse'] = df['type_route'].map(vitesses_par_type)

In [28]:
# Vérifier les nouvelles colonnes
print(df[['ville_depart', 'ville_arrivee', 'type_route', 'vitesse']].head())

  ville_depart ville_arrivee  type_route  vitesse
0       Agadir         Rabat           2      100
1   Casablanca     Marrakech           2      100
2       Tanger            F?           1       70
3        Oujda         Nador           1       70
4    Marrakech        Agadir           1       70


In [29]:
# Définir les colonnes d'entrée
features = ['distance', 'hour', 'weekday', 'is_rain', 'is_rush_hour', 'type_route', 'vitesse']
X = df[features]
y = df['trip_duration']
model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [36]:
# Sauvegarder dans un nouveau fichier CSV (ou écraser l'ancien)
df.to_csv("C:\\Users\\DELL\\Downloads\\data_set.csv", index=False, sep=';')
def prediction_personnalisee(model):
    print("\n=== Prédiction durée de trajet personnalisé ===")
    
    ville_depart = input("Ville de départ : ").strip()
    ville_arrivee = input("Ville d'arrivée : ").strip()
    distance = get_distance(ville_depart, ville_arrivee)
    if distance is None:
        print("Impossible de faire la prédiction sans distance.")
        return
    
    hour = int(input("Heure (0-23) : "))
    weekday = int(input("Jour de la semaine (0 = Lundi, ..., 6 = Dimanche) : "))
    is_rain = int(input("Pluie ? (0 = non, 1 = oui) : "))
     # Déterminer automatiquement si c’est une heure de pointe
    if (7 <= hour <= 9) or (16 <= hour <= 19):
        is_rush_hour = 1
    else:
        is_rush_hour = 0
    type_route = int(input("Type de route (0 = ville, 1 = route, 2 = autoroute) : "))
    
    # Calculer la vitesse moyenne en fonction du type de route
    vitesse = vitesses_par_type.get(type_route, 70)  # valeur par défaut : 70 km/h
    
    # Créer le DataFrame avec toutes les colonnes nécessaires
    input_data = pd.DataFrame([[distance, hour, weekday, is_rain, is_rush_hour, type_route, vitesse]],
                              columns=features)
    
    # Prédiction
    pred = model.predict(input_data)
    
    print(f"\n🟢 Durée estimée du trajet de {ville_depart} à {ville_arrivee} : {pred[0]:.2f} minutes")


In [41]:
# Appel de la fonction
prediction_personnalisee(model)


=== Prédiction durée de trajet personnalisé ===


Ville de départ :  Agadir
Ville d'arrivée :  Rabat
Heure (0-23) :  14
Jour de la semaine (0 = Lundi, ..., 6 = Dimanche) :  4
Pluie ? (0 = non, 1 = oui) :  1
Type de route (0 = ville, 1 = route, 2 = autoroute) :  2



🟢 Durée estimée du trajet de Agadir à Rabat : 612.17 minutes
